In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import random
import csv
import json

import numpy as np
import config

import yaml

load_dotenv(find_dotenv())
engine = create_engine(f'postgresql://{config.db_username}:{config.db_password}@{config.db_host}:{config.db_port}/{config.db_name}')
connection = engine.connect()

# Utils for migration

In [4]:
import time
import re

# Function to remove comments from SQL file
def remove_comments(sql_content):
    # Remove single-line comments (--) and multi-line comments (/* ... */)
    sql_content = re.sub(r'--.*', '', sql_content)  # Remove single-line comments
    sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)  # Remove block comments
    return sql_content


def initialize_db():
    with engine.connect() as connection:
        try:
            with connection.begin():
                connection.execute(text('DROP SCHEMA IF EXISTS tmp_migration_db CASCADE'))
                connection.execute(text('CREATE SCHEMA tmp_migration_db'))
                connection.execute(text('SET search_path TO tmp_migration_db, public'))
            # print("initialized to ",connection.execute(text('SHOW search_path')).fetchone())
        except Exception as e:
            print(f"Error executing initialization")


def simulate_migration(migration_file_name, num_sims):
    log_entries = []

    # Load and clean the SQL file
    with open(f'../migrations/{migration_file_name}.sql', 'r') as file:
        sql_content = file.read()

    # Remove comments and split into individual queries
    cleaned_sql = remove_comments(sql_content)
    queries = [query.strip() for query in cleaned_sql.split(';') if query.strip()]

    for i in range(num_sims):
        print(f'Run {i+1}/{num_sims}')
        initialize_db()
        
        # log_entry = dict()
        # log_entry['Run'] = i

        # Track the execution time
        start_time = time.time()

        # Execute each query in the file
        j = 1
        with engine.connect() as connection:
            with connection.begin():
                for query in queries:
                    try:
                        start_time_q = time.time()

                        connection.execute(text(query))
                        
                        end_time_q = time.time()
                        execution_time_q = end_time_q - start_time_q
                        # if query.startswith("CREATE TABLE"):
                        #     log_entry[f'Q{j}'] = execution_time_q
                        #     j += 1

                        # print(query)
                        # log_entry = rows[0][0]
                        # log_entry['QueryType'] = (chosen_index + 1)
                        # log_entry['NumRows'] = rows[0][0]["Plan"]["Actual Rows"]
                        # log_entry['FullQuery'] = pre_query
                        # log_entry['QueryParams'] = query_params

                        log_entries.append((i+1, j, execution_time_q, re.sub(r'[\n\t]', ' ', query)[:min(len(query), 100)]))  # Store run number, query number, execution time, and query snippet
                        j += 1
                    except Exception as e:
                        print(f"Error executing query: {query}\nError: {e}")

        # Calculate the total execution time
        end_time = time.time()
        execution_time = end_time - start_time
        
        # log_entry['Time'] = execution_time
        # log_entries.append(log_entry)

    log_df = pd.DataFrame(log_entries, columns=['Run', 'QueryNumber', 'ExecutionTime', 'QuerySnippet'])
    return log_df

# Execution

In [7]:
# scale = '1'
mig_conf = []
# mig_conf.append({ 'migration_file_name': '2003-Optimized' })
mig_conf.append({ 'migration_file_name': '2013-Optimized', 'scale': '1' })
mig_conf.append({ 'migration_file_name': '2023-Optimized', 'scale': '1' })
mig_conf.append({ 'migration_file_name': '2013-Optimized', 'scale': '2' })
mig_conf.append({ 'migration_file_name': '2023-Optimized', 'scale': '2' })
mig_conf.append({ 'migration_file_name': '2013-Optimized', 'scale': '5' })
mig_conf.append({ 'migration_file_name': '2023-Optimized', 'scale': '5' })
mig_conf.append({ 'migration_file_name': '2013-Optimized', 'scale': '10' })
mig_conf.append({ 'migration_file_name': '2023-Optimized', 'scale': '10' })

num_sims = 10

outdir = (f"../results")
if not os.path.exists(outdir):
    os.mkdir(outdir)

for i in range(len(mig_conf)):
    print(f'** RUNNING {mig_conf[i]['scale']}x/{mig_conf[i]['migration_file_name']} ({num_sims} migrations) **')
    log_df = simulate_migration(f'{mig_conf[i]['scale']}x/{mig_conf[i]["migration_file_name"]}', num_sims)
    # log_df.to_csv(f"{outdir}/tmp/results_mig_{mig_conf[i]['scale']}x_{mig_conf[i]['migration_file_name']}_detail.csv",index=False)
    log_df.to_excel(f"{outdir}/tmp/results_mig_{mig_conf[i]['scale']}x_{mig_conf[i]['migration_file_name']}_detail.xlsx", index=False)

** RUNNING 1x/2013-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 1x/2023-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 2x/2013-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 2x/2023-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 5x/2013-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 5x/2023-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 10x/2013-Optimized (10 migrations) **
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10
** RUNNING 10x/2023-Optimi

# Analysis

In [26]:
f = {
    '2013': 1.74,
    '2023': 0.72,
}
oldQTime = {
    '2013': {
        1: 63.9,
        2: 59.8,
        5: 62.7,
        10: 63.8
    },
    '2023': {
        1: 269.7,
        2: 490.9,
        5: 1134.6,
        10: 2304.3
    }
}

import pandas as pd
outdir = (f"../results")
y = '2013'
file = f"results_mig_{y}-Optimized_detail.xlsx"

# read exccel file in outdir = (f"../results"), sheet1 
df = pd.read_excel(f"{outdir}/{file}", sheet_name='MigTimes')

# aggegate by TableNumber,Scale,Run to get the sum of ExecutionTime
df = df.groupby(['TableNumber', 'Scale', 'Run'])[['ExecutionTime']].sum().reset_index()

# aggregate by TableNumber,Scale to get the mean of ExecutionTime
df = df.groupby(['TableNumber', 'Scale'])[['ExecutionTime']].mean().reset_index()

# For every scale, calculate the prefix sum of ExecutionTime over the TableNumber
df['MigTime'] = df.groupby('Scale')['ExecutionTime'].cumsum()

# Add one row per Scale (1,2,5,10) with TableNumber 0, ExecutionTime 0, and PrefixSum 0
scales = df['Scale'].unique()
new_rows = pd.DataFrame({'TableNumber': 0, 'Scale': scales, 'ExecutionTime': 0, 'MigTime': 0})
df = pd.concat([df, new_rows], ignore_index=True)

# Sort by Scale and TableNumber
df = df.sort_values(by=['Scale', 'TableNumber'])
# print(df)

dfq = pd.read_excel(f"{outdir}/{file}", sheet_name='QueryTimes')

def getTimesFromDfq(row):
    global dfq
    # Filter dfq for the current scale
    dfq_scale = dfq[dfq['Scale'] == row['Scale']]

    f = 'MinMigTableForNew'
    # f = 'MinMigTableForNew2'
    
    dfq_scale_old = dfq_scale[dfq_scale[f] > row['TableNumber']]
    dfq_scale_new = dfq_scale[dfq_scale[f] <= row['TableNumber']]
    # get sum of ExecutionTimeOld*Frequency from dfq_scale_old
    execution_time_old = (dfq_scale_old['ExecutionTimeOld'] * dfq_scale_old['Frequency']).sum()
    # get sum of ExecutionTimeNew*Frequency from dfq_scale_new
    execution_time_new = (dfq_scale_new['ExecutionTimeNew'] * dfq_scale_new['Frequency']).sum()

    return round(execution_time_old + execution_time_new,1)

# for every TableNumber and Scale in df, get the sum of execution times from dfq where Scales match; if MinMigTableForNew is lower than TableNumber, then consider ExecutionTimeOld*Frequency from dfq, else consider ExecutionTimeNew*Frequency
df['TotQTime']=df.apply(lambda row: getTimesFromDfq(row), axis=1)
df['TimeToPassMigTime']=df.apply(lambda row: row['MigTime']*f[y]*1000/(oldQTime[y][row['Scale']]-row['TotQTime']) if oldQTime[y][row['Scale']]-row['TotQTime']!=0 else None, axis=1)
df['TimeGain'] = df.apply(lambda row: oldQTime[y][row['Scale']]-row['TotQTime'], axis=1)
# exclude ExecutionTime from df
df = df.drop(columns=['ExecutionTime'])

#rename columns of df
df = df.rename(columns={'TableNumber': 'MigQuery', 'MigTime': 'MigTime (ms)', 'TotQTime': 'Avg Q Time (ms)', 'TimeToPassMigTime': 'TimeToPassMigTime (s)'})

# print(df)
df.to_excel(f"{outdir}/tmp/results_mig_{y}_analysis.xlsx", index=False)